In [1]:
import xarray as xr
import torch
import sys

sys.path.append("../src")
from constants import INPT_VARS, EXTRA_VARS, OUT_VARS
import os
import numpy as np
from einops import rearrange
from utils.train_utils import decomposed_mse, decomposed_mse_diff_weighted, decomposed_mse_cos_weighted, SmoothedValue, MetricLogger, extract_wet, extract_surface_wet

In [2]:
data_dir = "/pscratch/sd/s/suryad/data"
wet_file = "OM4_5daily_v0.2.1_wetmask"
surface_wet_file = "surface_wet.pt"
data_zarr = "3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975"
data_means_zarr = "3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_means"
data_stds_zarr = "3D_data_OM4_5daily_v0.2.1_with_hfds_anom_1975_stds"
grid_file = "Grid_New.nc"

inputs = INPT_VARS["3D_onlyTemp_all"]
extra_in = EXTRA_VARS["3D_all"]
outputs = OUT_VARS["3D_onlyTemp_all"]
lag = 1
interval = 1
hist = 1
N_samples = 20
N_val = 10
steps = 4

s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val


data = xr.open_zarr(os.path.join(data_dir, data_zarr))
data_mean = xr.open_zarr(os.path.join(data_dir, data_means_zarr))
data_std = xr.open_zarr(os.path.join(data_dir, data_stds_zarr))

wet_zarr = xr.open_zarr(os.path.join(data_dir, wet_file))
wet = extract_wet(wet_zarr, outputs, hist)
surface_wet = extract_surface_wet(wet_zarr)

torch.Size([38, 180, 360])


### Data Loading

#### data_CNN_Disk

In [3]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
import torch.utils.data as data
from scipy.ndimage import gaussian_filter
from einops import rearrange
import os


class data_CNN_Disk(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        ind_start,
        long_rollout,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.ind_start = ind_start

        assert self.interval == 1
        assert self.lag == 1

        data = data.isel(time=slice(self.ind_start, None))
        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]

        # This class will be used only for validation and rollouts
        # Rolling indices to keep track of histories/past states:
        # HIST=0 ; 0->[0, 1]; 1->[1, 2]; 2->[2, 3]; 3->[3, 4]
        # HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[2, 3], [4, 5]]; 2->[[4, 5], [6, 7]]; 3->[[6, 7], [8, 9]]
        # HIST=2 ; 0->[[0, 1, 2], [3, 4, 5]]; 1->[[3, 4, 5], [6, 7, 8]]; 2->[[6, 7, 8], [9, 10, 11]]; 3->[[9, 10, 11], [12, 13, 14]]
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        total_steps = 2 * self.hist + 1
        rolling_indices = (
            indices.rolling(time=len(data.time) - total_steps, center=False)
            .construct("window_dim")
            .astype(int)
        )
        rolling_indices = rolling_indices.transpose("window_dim", "time").isel(
            time=slice(len(data.time) - total_steps - 1, None)
        )  # Remove first few null indices
        self.rolling_indices = rolling_indices.isel(
            window_dim=slice(0, None, self.hist + 1)
        )  # Skip indices based on history

        if long_rollout:
            window0 = self.rolling_indices.isel(window_dim=0)
            print(
                "Long rollout will begin with input and produce output from time index {0} and {1} respectively".format(
                    window0.isel(time=0).values + ind_start,
                    window0.isel(time=self.hist + 1).values + ind_start,
                )
            )

        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]
        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]
        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if type(idx) == slice:
            if idx.start == None and idx.stop == None:
                idx = slice(0, self.size, idx.step)
            elif idx.start == None:
                idx = slice(0, idx.stop, idx.step)
            elif idx.stop == None:
                idx = slice(idx.start, self.size, idx.step)
        elif type(idx) == int:
            idx = slice(idx, idx + 1, 1)

        rolling_idx = self.rolling_indices.isel(window_dim=idx)
        x_index = xr.Variable(
            ["window_dim", "time"], rolling_idx
        )
        data_in = self.inputs_no_extra.isel(time=x_index).isel(
            time=slice(None, self.hist + 1)
        )
        data_in = (
            (data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std
        ).fillna(0)
        data_in = (
            data_in.to_array()
            .transpose("window_dim", "time", "variable", "y", "x")
            .to_numpy()
        )
        # data_in = rearrange(
        #     data_in, "window_dim time variable y x -> window_dim (time variable) y x"
        # )
        data_in_boundary = self.extras.isel(time=x_index).isel(time=slice(None, self.hist + 1))
        data_in_boundary = (
            (data_in_boundary - self.extras_mean) / self.extras_std
        ).fillna(0)
        data_in_boundary = (
            data_in_boundary.to_array()
            .transpose("window_dim", "time", "variable", "y", "x")
            .to_numpy()
        )
        data_in = np.concatenate((data_in, data_in_boundary), axis=2)

        label = self.outputs.isel(time=x_index).isel(time=slice(self.hist + 1, None))
        label = ((label - self.out_mean) / self.out_std).fillna(0)
        label = (
            label.to_array()
            .transpose("window_dim", "time", "variable", "y", "x")
            .to_numpy()
        )
        # label = rearrange(
        #     label, "window_dim time variable y x -> window_dim (time variable) y x"
        # )

        items = (torch.from_numpy(data_in).float(), torch.from_numpy(label).float())

        return items

In [9]:
data = xr.open_zarr(os.path.join(data_dir, data_zarr))
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=True,
    device="cuda",
)
d = val_data[0]
print(d[0].shape)
print(d[1].shape)
d = val_data[1]
d = val_data[len(val_data) - 1]
# v = val_data[0]
# print(x_index)

# v, x_index = val_data[1]
# print(x_index)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 21 and 23 respectively
torch.Size([1, 2, 22, 180, 360])
torch.Size([1, 2, 19, 180, 360])


In [5]:
print(len(val_data))

10


In [14]:
assert v[0].shape[1] == (
    v[1].shape[1] + 3
), f"Input shape {v[0].shape} and output shape {v[1].shape} do not match"

In [70]:
hist = 0
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=False,
    device="cuda",
)

old_val_data = old_data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    e_train,
    device="cuda",
)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [71]:
assert (old_val_data[:][1] == val_data[:][1]).all()
assert (old_val_data[:][0] == val_data[:][0]).all()

In [72]:
%%time
d0 = val_data[6]
print(d0[0].shape)
print(d0[1].shape)

torch.Size([1, 80, 180, 360])
torch.Size([1, 77, 180, 360])
CPU times: user 3.7 s, sys: 418 ms, total: 4.12 s
Wall time: 2.6 s


In [73]:
%%time
s = slice(6, 10)
d1 = val_data[s]
print(d1[0].shape)
print(d1[1].shape)

torch.Size([4, 80, 180, 360])
torch.Size([4, 77, 180, 360])
CPU times: user 4.95 s, sys: 1.52 s, total: 6.47 s
Wall time: 3.52 s


In [74]:
# Assertion
idx = s
ind_out = slice(
    idx.start + val_data.hist + val_data.lag,
    idx.stop * val_data.interval + val_data.hist + val_data.lag,
    val_data.interval,
)
assert (
    np.nansum(
        val_data.outputs.isel(time=ind_out).to_array().to_numpy()
        - val_data.outputs.isel(
            time=xr.Variable(
                ["window_dim", "time"], val_data.rolling_indices.isel(window_dim=idx)
            )
        )
        .isel(time=-1)
        .to_array()
        .to_numpy()
    )
    == 0.0
)

In [75]:
assert (d1[0][2][:77] == d1[1][1]).all()
assert (d0[1] == d1[1][0]).all()

In [112]:
hist = 1
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=False,
    device="cuda",
)
d = val_data[0]
d = val_data[1]
d = val_data[len(val_data) - 1]

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


<xarray.DataArray (window_dim: 1, time: 4)>
array([[0, 1, 2, 3]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[2, 3, 4, 5]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[18, 19, 20, 21]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim


In [77]:
%%time
d2 = val_data[6:10]
print(d2[0].shape)
print(d2[1].shape)

torch.Size([4, 157, 180, 360])
torch.Size([4, 154, 180, 360])
CPU times: user 6.08 s, sys: 2.51 s, total: 8.59 s
Wall time: 3.84 s


In [78]:
# HIST=0 ; 0->[0, 1]; 1->[1, 2]; 2->[2, 3]; 3->[3, 4]
# HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[1, 2], [3, 4]]; 2->[[2, 3], [4, 5]]; 3->[[3, 4], [5, 6]]

assert (d2[0][0][:77] == d1[0][0][:77]).all()
assert (d2[0][0][77:154] == d1[1][0]).all()
assert (d2[1][0][:77] == d1[1][1]).all()
assert (d2[1][0][77:154] == d1[1][2]).all()

#### data_CNN_Disk_steps

old

In [53]:
class old_data_CNN_Disk_steps(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        steps,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.steps = steps

        assert self.interval == 1
        assert self.lag == 1

        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]

        # This class will be used only for training
        # Rolling indices to keep track of histories/past states (without skips):
        # HIST=0, 4 steps ; 0->[0in, 1out, 1in, 2out, 2in, 3out, 3in, 4out]
        # HIST=1, 4 steps , 0->[[0in, 1in], [2out, 3out], [2in, 3in], [4out, 5out]], 1->[[1in, 2in], [3out, 4out], [3in, 4in], [5out, 6out]]
        # HIST=2, 4 steps , 0->[[0in, 1in, 2in], [3out, 4out, 5out], [3in, 4in, 5in], [6out, 7out, 8out], [6in, 7in, 8in], [9out, 10out, 11out], [9in, 10in, 11in], [12out, 13out, 14out]]
        indices = xr.DataArray(
            np.arange(data.time.size),
            dims=["time"],
            coords={"time": data.time},
        )
        total_steps = 2 * self.hist + 1
        rolling_indices = (
            indices.rolling(time=len(data.time) - total_steps, center=False)
            .construct("window_dim")
            .astype(int)
        )
        self.rolling_indices = rolling_indices.transpose("window_dim", "time").isel(
            time=slice(len(data.time) - total_steps - 1, None)
        )  # Remove first few null indices

        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]
        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        return self.size - self.steps * (self.hist + 1) - self.hist

    def __getitem__(self, idx):
        outputs = []
        
        if idx >= len(self):
            raise IndexError("Index out of range")

        assert type(idx) == int
        for step in range(self.steps):
            start = idx + step * (self.hist + 1)
            end = start + 1
            idx_slice = slice(
                start, end, self.interval
            )  # Create a slice for similar indexing as in data_CNN_Disk
            rolling_idx = self.rolling_indices.isel(window_dim=idx_slice)
            print(rolling_idx)
            x_index = xr.Variable(
                ["window_dim", "time"], rolling_idx
            )
            data_in = self.inputs_no_extra.isel(time=x_index).isel(
                time=slice(None, self.hist + 1)
            )
            data_in = (
                (data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std
            ).fillna(0)
            data_in = (
                data_in.to_array()
                .transpose("window_dim", "time", "variable", "y", "x")
                .to_numpy()
            )
            data_in = rearrange(
                data_in,
                "window_dim time variable y x -> window_dim (time variable) y x",
            )
            data_in_boundary = self.extras.isel(time=x_index).isel(time=self.hist)
            data_in_boundary = (
                (data_in_boundary - self.extras_mean) / self.extras_std
            ).fillna(0)
            data_in_boundary = (
                data_in_boundary.to_array()
                .transpose("window_dim", "variable", "y", "x")
                .to_numpy()
            )
            data_in = np.concatenate((data_in, data_in_boundary), axis=1).squeeze()

            label = self.outputs.isel(time=x_index).isel(
                time=slice(self.hist + 1, None)
            )
            label = ((label - self.out_mean) / self.out_std).fillna(0)
            label = (
                label.to_array()
                .transpose("window_dim", "time", "variable", "y", "x")
                .to_numpy()
            )
            label = rearrange(
                label, "window_dim time variable y x -> window_dim (time variable) y x"
            ).squeeze()

            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs


New

In [104]:
hist = 1 
total_steps = (2 * hist + 2)
stride = 6

# Calculate the number of windows
num_windows = data.time.size - (total_steps - 1) * stride

# Create base indices
indices = np.arange(num_windows)
indices_da = xr.DataArray(
    indices,
    dims=["window_dim"],
    coords={"window_dim": np.arange(num_windows)},
)

# Create window dimension
window_dim = xr.DataArray(
    np.arange(total_steps),
    dims=["time"],
)

# Construct rolling indices
rolling_indices = indices_da + stride * window_dim

# Display the rolling indices
rolling_indices

train_size = (500 - steps * (hist + 1) * stride - hist * stride)
for i in range(train_size, train_size+steps*(hist+1)*stride, (hist+1)*stride):
    print(rolling_indices[i])


<xarray.DataArray (time: 4)>
array([446, 452, 458, 464])
Coordinates:
    window_dim  int64 446
Dimensions without coordinates: time
<xarray.DataArray (time: 4)>
array([458, 464, 470, 476])
Coordinates:
    window_dim  int64 458
Dimensions without coordinates: time
<xarray.DataArray (time: 4)>
array([470, 476, 482, 488])
Coordinates:
    window_dim  int64 470
Dimensions without coordinates: time
<xarray.DataArray (time: 4)>
array([482, 488, 494, 500])
Coordinates:
    window_dim  int64 482
Dimensions without coordinates: time


In [15]:
class data_CNN_Disk_steps(torch.utils.data.Dataset):

    def __init__(
        self,
        data,
        inputs_str,
        extra_in_str,
        outputs_str,
        wet,
        data_mean,
        data_std,
        n_samples,
        lag,
        interval,
        hist,
        steps,
        stride=1,
        device="cuda",
    ):
        super().__init__()
        self.device = device

        self.size = n_samples
        self.lag = lag
        self.interval = interval
        self.hist = hist
        self.steps = steps
        self.stride = stride

        assert self.interval == 1
        assert self.lag == 1

        self.inputs = data[inputs_str + extra_in_str]
        self.outputs = data[outputs_str]
        self.inputs_no_extra = data[inputs_str]
        self.extras = data[extra_in_str]

        # This class will be used only for training
        total_steps = (2 * self.hist + 2)

        # Calculate the number of windows
        num_windows = data.time.size - (total_steps - 1) * self.stride

        # Create base indices
        indices = np.arange(num_windows)
        indices_da = xr.DataArray(
            indices,
            dims=["window_dim"]
        )

        # Create window dimension
        window_dim = xr.DataArray(
            np.arange(total_steps),
            dims=["time"]
        )

        # Construct rolling indices
        self.rolling_indices = indices_da + stride * window_dim

        self.inputs_no_extra_mean = data_mean[inputs_str]
        self.inputs_no_extra_std = data_std[inputs_str]
        self.extras_mean = data_mean[extra_in_str]
        self.extras_std = data_std[extra_in_str]
        self.in_mean = data_mean[inputs_str + extra_in_str]
        self.in_std = data_std[inputs_str + extra_in_str]

        self.out_mean = data_mean[outputs_str]
        self.out_std = data_std[outputs_str]

        self.wet = wet

    def set_device(self, device):
        self.device = device

    def __len__(self):
        return (self.size - self.steps * (self.hist + 1) * self.stride - self.hist * self.stride)
    
    def __getitem__(self, idx):
        outputs = []
        
        if idx >= len(self):
            raise IndexError("Index out of range")

        assert type(idx) == int
        prev_rolling_idx = None
        for step in range(self.steps):
            start = idx + step * (self.hist + 1) * self.stride
            end = start + 1
            idx_slice = slice(
                start, end, self.interval
            )  # Create a slice for similar indexing as in data_CNN_Disk
            rolling_idx = self.rolling_indices.isel(window_dim=idx_slice)
            if prev_rolling_idx is not None:
                assert (prev_rolling_idx.isel(time=slice(self.hist+1, None)) - rolling_idx.isel(time=slice(0, self.hist+1))).sum() == 0 # Prev output = Cur Input 
                assert (rolling_idx.diff("time") == self.stride).all() # Stride is maintained
                assert rolling_idx.isel(time=-1) < self.size # Last index check
            x_index = xr.Variable(
                ["window_dim", "time"], rolling_idx
            )
            data_in = self.inputs_no_extra.isel(time=x_index).isel(
                time=slice(None, self.hist + 1)
            )
            data_in = (
                (data_in - self.inputs_no_extra_mean) / self.inputs_no_extra_std
            ).fillna(0)
            data_in = (
                data_in.to_array()
                .transpose("window_dim", "time", "variable", "y", "x")
                .to_numpy()
            )
            # data_in = rearrange(
            #     data_in,
            #     "window_dim time variable y x -> window_dim (time variable) y x",
            # )
            data_in_boundary = self.extras.isel(time=x_index).isel(time=slice(None, self.hist + 1))
            data_in_boundary = (
                (data_in_boundary - self.extras_mean) / self.extras_std
            ).fillna(0)
            data_in_boundary = (
                data_in_boundary.to_array()
                .transpose("window_dim", "time", "variable", "y", "x")
                .to_numpy()
            )
            data_in = np.concatenate((data_in, data_in_boundary), axis=2).squeeze()

            label = self.outputs.isel(time=x_index).isel(
                time=slice(self.hist + 1, None)
            )
            label = ((label - self.out_mean) / self.out_std).fillna(0)
            label = (
                label.to_array()
                .transpose("window_dim", "time", "variable", "y", "x")
                .to_numpy()
            ).squeeze()
            # label = rearrange(
            #     label, "window_dim time variable y x -> window_dim (time variable) y x"
            # )

            outputs.append(torch.from_numpy(data_in).float())
            outputs.append(torch.from_numpy(label).float())

        return outputs

In [16]:
lag = 1
interval = 1
hist = 1
N_samples = 500
N_val = 10
steps = 4

s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val

train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    stride=3,
    device="cuda",
)

d = train_data[0]
# d = train_data[len(train_data) - 1]
print(d[0].shape)
print(d[1].shape)
print(len(train_data))

torch.Size([2, 22, 180, 360])
torch.Size([2, 19, 180, 360])
473


In [108]:
train_data = old_data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)

d = train_data[10]
print(d[0].shape)
print(d[1].shape)
print(len(train_data))

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


<xarray.DataArray (window_dim: 1, time: 4)>
array([[10, 11, 12, 13]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[12, 13, 14, 15]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[14, 15, 16, 17]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[16, 17, 18, 19]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
torch.Size([41, 180, 360])
torch.Size([38, 180, 360])
11


In [100]:
assert (d[0] == d0[0]).all()
assert (d[1] == d0[1]).all()

In [101]:
for i in range(len(d)):
    if i % 2 == 0:
        assert (d0[i][:77] == d[i][:77]).all()
    else:
        assert (d0[i] == d[i]).all()

In [129]:
hist = 2
train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [130]:
d2 = train_data[len(train_data)-1]
print(d2[0].shape)
print(d2[1].shape)

<xarray.DataArray (window_dim: 1, time: 6)>
array([[ 5,  6,  7,  8,  9, 10]])
Coordinates:
  * time     (time) object 2022-12-04 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 6)>
array([[ 8,  9, 10, 11, 12, 13]])
Coordinates:
  * time     (time) object 2022-12-04 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 6)>
array([[11, 12, 13, 14, 15, 16]])
Coordinates:
  * time     (time) object 2022-12-04 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 6)>
array([[14, 15, 16, 17, 18, 19]])
Coordinates:
  * time     (time) object 2022-12-04 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
torch.Size([234, 180, 360])
torch.Size([231, 180, 360])


In [47]:
# HIST=0, 4 steps ; 0->[0in, 1out, 1in, 2out, 2in, 3out, 3in, 4out]
# HIST=1, 4 steps , 0->[[0in, 1in], [2out, 3out], [2in, 3in], [4out, 5out], [4in, 5in], [6out, 7out], [6in, 7in], [8out, 9out]]
assert (d0[0][:77] == d2[0][:77]).all()
assert (d0[1] == d2[0][77:154]).all()
assert (d0[3] == d2[1][:77]).all()
assert (d0[5] == d2[1][77:]).all()
assert (d0[7][:77] == d2[3][:77]).all()

### Model Forward/Inference

In [6]:
def forward_once(input_tensor):
    return input_tensor[:, :output_channels]


def forward_once(input_tensor):
    return input_tensor[:, :output_channels]


def forward(
    inputs,
    output_only_last=False,
    loss_fn=None,
) -> torch.Tensor:
    outputs = []
    loss = None

    ### TEMP
    for n in range(len(inputs)):
        inputs[n] = inputs[n].unsqueeze(0)

    N, C, H, W = inputs[0].shape

    for step in range(len(inputs) // 2):
        print(step)
        if step == 0:
            input_tensor = inputs[0]
        else:
            inputs_0 = outputs[-1]
            print("inputs_0", inputs_0.shape, "outputs[]", -1)
            print(
                "inputs[2*step]",
                inputs[2 * step].shape,
                "Slice: ",
                output_channels,
                None,
            )
            input_tensor = torch.cat(
                [
                    inputs_0,
                    inputs[2 * step][:, output_channels:],
                ],
                dim=1,
            )

        print(input_tensor.shape)

        assert (
            input_tensor.shape[1] == input_channels
        ), f"Input shape is {input_tensor.shape[1]} but should be {input_channels}"
        decodings = forward_once(input_tensor)
        if pred_residuals:
            reshaped = (
                input_tensor[:, output_channels * hist : output_channels * (hist + 1)]
                + decodings
            )  # Residual prediction
        else:
            reshaped = decodings  # Absolute prediction

        if loss_fn is not None:
            assert (
                reshaped.shape == inputs[2 * step + 1].shape
            ), f"Output shape is {reshaped.shape} but should be {inputs[2 * step + 1].shape}"
            if loss is None:
                loss = loss_fn(
                    reshaped,
                    inputs[2 * step + 1],
                )
            else:
                loss += loss_fn(
                    reshaped,
                    inputs[2 * step + 1],
                )

        outputs.append(reshaped)

    if loss_fn is None:
        if output_only_last:
            res = outputs[-1]
        else:
            res = outputs
        return res

    else:
        return loss

In [8]:
hist = 1
output_channels = (hist + 1) * 77
input_channels = (hist + 1) * 77 + 3
pred_residuals = False

# HIST=0, 4 steps ; 0->[0in, 1out, 1in, 2out, 2in, 3out, 3in, 4out]
# HIST=1, 4 steps , 0->[[0in, 1in], [2out, 3out], [1in, 2in], [3out, 4out], [2in, 3in], [4out, 5out], [3in, 4in], [5out, 6out]]
# HIST=2, 4 steps , 0->[[0in, 1in, 2in], [3out, 4out, 5out], [1in, 2in, 3in], [4out, 5out, 6out], [2in, 3in, 4in], [5out, 6out, 7out], [3in, 4in, 5in], [6out, 7out, 8out]]
data = xr.open_zarr(os.path.join(data_dir, data_zarr))


train_data = data_CNN_Disk_steps(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_samples,
    lag,
    interval,
    hist,
    steps,
    device="cuda",
)
a = forward(train_data[2], output_only_last=False, loss_fn=torch.nn.MSELoss())

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


<xarray.DataArray (window_dim: 1, time: 4)>
array([[2, 3, 4, 5]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[4, 5, 6, 7]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[6, 7, 8, 9]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
<xarray.DataArray (window_dim: 1, time: 4)>
array([[ 8,  9, 10, 11]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
0
torch.Size([1, 157, 180, 360])
1
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -2
inputs[2*step] torch.Size([1, 157, 180, 360]) Slice:  154 None
torch.Size([1, 157, 180, 360])
2
inputs_0 torch.Size([1, 154, 180, 360]) 

In [38]:
# HIST=0 ; 0->[0, 1]; 1->[2, 3]; 2->[3, 4]; 3->[5, 6]
# HIST=1 ; 0->[[0, 1], [2, 3]]; 1->[[2, 3], [4, 5]]; 2->[[4, 5], [6, 7]]
# HIST=2 ; 0->[[0, 1, 2], [3, 4, 5]]; 1->[[2, 3, 4], [5, 6, 7]];]
def inference(
    inputs, num_steps=None, output_only_last=False, device="cuda"
) -> torch.Tensor:
    outputs = []
    for step in range(num_steps):
        print(step)
        if step == 0:
            input_tensor = inputs[0][0].to(
                device=device
            )  # inputs[0][0] is the input at step 0
        else:
            inputs_0 = outputs[-1].unsqueeze(
                0
            )  # Last output corresponding to current input
            print("inputs_0", inputs_0.shape, "outputs[]", -1)
            print(
                "inputs[step][0][:, output_channels :]",
                inputs[step][0][:, output_channels:].shape,
                "Slice: ",
                output_channels,
                None,
            )
            input_tensor = torch.cat(
                [
                    inputs_0,
                    inputs[step][0][
                        :, output_channels:
                    ].to(  # concatenate the boundary conditions
                        device=device
                    ),
                ],
                dim=1,
            )
        print(input_tensor.shape)

        assert (
            input_tensor.shape[1] == input_channels
        ), f"Input shape is {input_tensor.shape[1]} but should be {input_channels}"
        decodings = forward_once(input_tensor)
        if pred_residuals:
            reshaped = input_tensor[
                0, output_channels * hist : output_channels * (hist + 1)
            ].to(  # Residuals on last state in input
                device=device
            ) + decodings.squeeze(
                0
            )
        else:
            reshaped = decodings.squeeze(0)

        outputs.append(reshaped)

    if output_only_last:
        res = outputs[-1]
    else:
        res = outputs

    return res

In [39]:
hist = 2
s_train = lag * hist
e_train = s_train + N_samples * interval
e_test = e_train + interval * N_val
output_channels = (hist + 1) * 77
input_channels = (hist + 1) * 77 + 3
pred_residuals = False
data = xr.open_zarr(os.path.join(data_dir, data_zarr))
val_data = data_CNN_Disk(
    data,
    inputs,
    extra_in,
    outputs,
    wet,
    data_mean,
    data_std,
    N_val,
    lag,
    interval,
    hist,
    e_train,
    long_rollout=True,
    device="cuda",
)
print(val_data.rolling_indices)
inf = inference(val_data, num_steps=10, output_only_last=False, device="cpu")

/global/common/software/m4331/suryad/conda/emulator/lib/python3.10/site-packages/xarray/core/duck_array_ops.py:188: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


Long rollout will begin with input and produce output from time index 21 and 23 respectively
<xarray.DataArray (window_dim: 2361, time: 4)>
array([[   0,    1,    2,    3],
       [   2,    3,    4,    5],
       [   4,    5,    6,    7],
       ...,
       [4716, 4717, 4718, 4719],
       [4718, 4719, 4720, 4721],
       [4720, 4721, 4722, 4723]])
Coordinates:
  * time     (time) object 2022-12-14 12:00:00 ... 2022-12-29 12:00:00
Dimensions without coordinates: window_dim
0
torch.Size([1, 157, 180, 360])
1
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -1
inputs[step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  154 None
torch.Size([1, 157, 180, 360])
2
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -1
inputs[step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  154 None
torch.Size([1, 157, 180, 360])
3
inputs_0 torch.Size([1, 154, 180, 360]) outputs[] -1
inputs[step][0][:, output_channels :] torch.Size([1, 3, 180, 360]) Slice:  154 None
torch.Size

In [1]:
import torch
import torch.nn as nn
from thop import profile

# Simple 3-layer Conv2D network
class SimpleConv2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SimpleConv2D, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(128, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = torch.relu(x)
        x = self.conv3(x)
        return x


# Simple 3-layer Conv3D network
class SimpleConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SimpleConv3D, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, kernel_size=(1,3,3), stride=1, padding=(0,1,1))
        self.conv2 = nn.Conv3d(64, 128, kernel_size=(1,3,3), stride=1, padding=(0,1,1))
        self.conv3 = nn.Conv3d(128, out_channels, kernel_size=(1,3,3), stride=1, padding=(0,1,1))

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.conv2(x)
        x = torch.relu(x)
        x = self.conv3(x)
        return x


# Create instances of the models
model_2d = SimpleConv2D(in_channels=6, out_channels=6)
model_3d = SimpleConv3D(in_channels=3, out_channels=3)

# Example input for Conv2D model: [Batch, Channels, Height, Width]
input_2d = torch.randn(1, 6, 128, 128)

# Example input for Conv3D model: [Batch, Channels, Depth, Height, Width]
input_3d = torch.randn(1, 3, 2, 128, 128)

# Calculate FLOPs and Parameters using thop
flops_2d, params_2d = profile(model_2d, inputs=(input_2d,))
flops_3d, params_3d = profile(model_3d, inputs=(input_3d,))

# Print the results
print(f"2D ConvNet: FLOPs: {flops_2d / 1e9:.2f} GFLOPs, Params: {params_2d}M")
print(f"3D ConvNet: FLOPs: {flops_3d / 1e9:.2f} GFLOPs, Params: {params_3d}M")

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv3d'>.
2D ConvNet: FLOPs: 1.38 GFLOPs, Params: 84294.0M
3D ConvNet: FLOPs: 2.59 GFLOPs, Params: 79107.0M


In [18]:
import torch
import torch.nn as nn
from thop import profile

# Simple 3-layer Conv2D network
class SimpleConv2D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SimpleConv2D, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(128, out_channels, kernel_size=3, stride=1, padding=1)

    def forward(self, x):
        print(f"Input shape to conv1: {x.shape}, Kernel shape: {self.conv1.weight.shape}")
        x = self.conv1(x)
        x = torch.relu(x)

        print(f"Input shape to conv2: {x.shape}, Kernel shape: {self.conv2.weight.shape}")
        x = self.conv2(x)
        x = torch.relu(x)

        print(f"Input shape to conv3: {x.shape}, Kernel shape: {self.conv3.weight.shape}")
        x = self.conv3(x)
        
        print(f"Final output shape: {x.shape}")

        return x


# Simple 3-layer Conv3D network
class SimpleConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(SimpleConv3D, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, 64, kernel_size=(2,3,3), stride=1, padding=(0,1,1))
        self.conv2 = nn.Conv3d(64, 128, kernel_size=(1,3,3), stride=1, padding=(0,1,1))
        self.conv3 = nn.Conv3d(128, out_channels*2, kernel_size=(1,3,3), stride=1, padding=(0,1,1))

    def forward(self, x):
        print(f"Input shape to conv1: {x.shape}, Kernel shape: {self.conv1.weight.shape}")
        x = self.conv1(x)
        x = torch.relu(x)

        print(f"Input shape to conv2: {x.shape}, Kernel shape: {self.conv2.weight.shape}")
        x = self.conv2(x)
        x = torch.relu(x)

        print(f"Input shape to conv3: {x.shape}, Kernel shape: {self.conv3.weight.shape}")
        x = self.conv3(x)
        
        print(f"Final output shape: {x.shape}")

        return x


# Create instances of the models
model_2d = SimpleConv2D(in_channels=6, out_channels=6)
model_3d = SimpleConv3D(in_channels=3, out_channels=3)

# Example input for Conv2D model: [Batch, Channels, Height, Width]
input_2d = torch.randn(1, 6, 128, 128)

# Example input for Conv3D model: [Batch, Channels, Depth, Height, Width]
input_3d = torch.randn(1, 3, 2, 128, 128)

# Calculate FLOPs and Parameters using thop
flops_2d, params_2d = profile(model_2d, inputs=(input_2d,))
flops_3d, params_3d = profile(model_3d, inputs=(input_3d,))

# Print the results
print(f"2D ConvNet: FLOPs: {flops_2d / 1e9:.10f} GFLOPs, Params: {params_2d}")
print(f"3D ConvNet: FLOPs: {flops_3d / 1e9:.10f} GFLOPs, Params: {params_3d}")
print("Flops equal? ", flops_2d == flops_3d)
print("Params equal? ", params_2d == params_3d)

[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv3d'>.
2D ConvNet: FLOPs: 1.3778288640 GFLOPs, Params: 84294.0
3D ConvNet: FLOPs: 1.3778288640 GFLOPs, Params: 84294.0
Flops equal?  True
Params equal?  True
System memory used: 0.1068 GB
GPU memory used: 0.0000 GB
